# ADB Phase 2 Project Evaluation Notebook


**Purpose**: This notebook evaluates the performance of a semantic search project by analyzing databases of various sizes.

### Evaluation Focus:
- **Database Sizes**:
  - 1 Million Records
  - 10 Million Records
  - 15 Million Records
  - 20 Million Records

For each database size, this notebook will:
- Generate random vectors for the database.
- Use the `VecDB` class (implemented by students) to retrieve queries
- Evaluate and report retrieval time, accuracy, and RAM usage.

### Project Constraints:
Refer to the project document for details on RAM, Disk, Time, and Score constraints.

### Notebook Structure:
1. **Part 1 - Modifiable Cells**:
   - Includes cells that teams are allowed to modify, specifically for these variables only:
     - GitHub repository link (including PAT token).
     - Google Drive IDs for indexes files.
     - Paths for loading existing indexes.

2. **Part 2 - Non-Modifiable Cells**:
   - Contains essential setup and evaluation code that must not be modified.
   - Students should only modify inputs in Part 1 to ensure smooth execution of the notebook.

## Part 1 - Modifiable Cells

Each team must provide a unique GitHub repository link that includes a PAT token. This link will allow the notebook to download the necessary code for evaluation.

In [1]:
# !git clone https://github.com/farah-moh/vec_db.git

# Database Path Instructions


Teams need to specify paths for each database (1M, 10M, 15M, 20M records) as follows:

1. Zip each database directory/file after generation.
2. Upload the zip file to Google Drive.
3. Share the file with "Anyone with the link."
4. Extract the file ID from the link (e.g., for `https://drive.google.com/file/d/1j1gAU3kvdRqcOoKI5K5FgMMUZpOQANah/view`, the ID is `1j1gAU3kvdRqcOoKI5K5FgMMUZpOQANah`).
5. Assign each ID to the appropriate variable in Part 1.
6. Provide the local PATH for each database to be passed to the initializer for automatic loading of the database and index (to be submitted during the project final phase).

**Note**: The code will download and unzip these files automatically. Once extracted, the local path for each database should be specified to enable the notebook to load databases and indexes.

In [2]:
# TEAM_NUMBER = 1
# GDRIVE_ID_DB_1M = "1XbEP6sU0k0UbuPcQLkHJP7cdPtebpsbD"
# GDRIVE_ID_DB_10M = "1Jho9rair77eWdA5-iI_qDT2spJNesuDe"
# GDRIVE_ID_DB_15M = "1sPIgNxIuNDUUBnTvAuPMLryd1mqmgwV3"
# GDRIVE_ID_DB_20M = "1j1gAU3kvdRqcOoKI5K5FgMMUZpOQANah"
# PATH_DB_1M = "saved_db_1m.csv"
# PATH_DB_10M = "saved_db_10m.csv"
# PATH_DB_15M = "saved_db_15m.csv"
# PATH_DB_20M = "saved_db_20m.csv"

**Query Seed Number**:
This number will be adjusted during discussions by the instructor.


In [3]:
QUERY_SEED_NUMBER = 10

**Final Submission Checklist**:
Ensure the following items are included in your final submission:
- `TEAM_NUMBER`
- GitHub clone link (with PAT token)
- Google Drive IDs for each database:
  - `GDRIVE_ID_DB_1M`, `GDRIVE_ID_DB_10M`, `GDRIVE_ID_DB_15M`, `GDRIVE_ID_DB_20M`
- Paths for each database:
  - `PATH_DB_1M`, `PATH_DB_10M`, `PATH_DB_15M`, `PATH_DB_20M`
- Project document detailing the work and findings.

## Part 2: Do Not Modify Beyond This Point
### Note:
This section contains setup and evaluation code that should not be edited by students. Only the instructor may modify this section in case of a major bug.


In [4]:
# %load_ext autoreload
# %autoreload 2

In [5]:
# %cd vec_db

This cell to run any additional requirement that your code need <br>


In [6]:
# !conda install -y gdown  &> log.txt

In [7]:
!pip install memory-profiler &> log.txt
# !pip install -r requirements.txt

This cell to download the zip files and unzip them here.

In [8]:
# !gdown $GDRIVE_ID_DB_1M -O saved_db_1m.zip
# !gdown $GDRIVE_ID_DB_10M -O saved_db_10m.zip
# !gdown $GDRIVE_ID_DB_15M -O saved_db_15m.zip
# !gdown $GDRIVE_ID_DB_20M -O saved_db_20m.zip
# !unzip saved_db_1m.zip
# !unzip saved_db_10m.zip
# !unzip saved_db_15m.zip
# !unzip saved_db_20m.zip

These are the functions for running and reporting

In [9]:
import numpy as np
DB_SEED_NUMBER = 42
ELEMENT_SIZE = np.dtype(np.float32).itemsize
DIMENSION = 70

In [135]:
from typing import Dict, List, Annotated
import numpy as np
import os
import shutil
from sklearn.cluster import MiniBatchKMeans
import pickle
import heapq
import numpy as np

DB_SEED_NUMBER = 42
ELEMENT_SIZE = np.dtype(np.float32).itemsize
DIMENSION = 70


n_clusters_1 = 500
batch_size_1 = 10
nprobe_1 = 20

n_clusters_2 = 100
batch_size_2 = 10
nprobe_2 = 10



class VecDB:
    def __init__(self, database_file_path = "saved_db.dat", index_file_path = "index.dat", new_db = True, db_size = None) -> None:
        self.db_path = database_file_path
        self.index_path = index_file_path
        if new_db:
            if db_size is None:
                raise ValueError("You need to provide the size of the database")
            # delete the old DB file if exists
            if os.path.exists(self.db_path):
                os.remove(self.db_path)
            self.vectors = self.generate_database(db_size)

    def generate_database(self, size: int) -> None:
        rng = np.random.default_rng(DB_SEED_NUMBER)
        vectors = rng.random((size, DIMENSION), dtype=np.float32)
        self._write_vectors_to_file(vectors)
        self._build_index()
        return vectors

    def _write_vectors_to_file(self, vectors: np.ndarray) -> None:
        mmap_vectors = np.memmap(self.db_path, dtype=np.float32, mode='w+', shape=vectors.shape)
        mmap_vectors[:] = vectors[:]
        mmap_vectors.flush()

    def _get_num_records(self) -> int:
        return os.path.getsize(self.db_path) // (DIMENSION * ELEMENT_SIZE)

    def insert_records(self, rows: Annotated[np.ndarray, (int, 70)]):
        num_old_records = self._get_num_records()
        num_new_records = len(rows)
        full_shape = (num_old_records + num_new_records, DIMENSION)
        mmap_vectors = np.memmap(self.db_path, dtype=np.float32, mode='r+', shape=full_shape)
        mmap_vectors[num_old_records:] = rows
        mmap_vectors.flush()
        #TODO: might change to call insert in the index, if you need
        self._build_index()

    def get_one_row(self, row_num: int) -> np.ndarray:
        # This function is only load one row in memory
        try:
            offset = row_num * DIMENSION * ELEMENT_SIZE
            mmap_vector = np.memmap(self.db_path, dtype=np.float32, mode='r', shape=(1, DIMENSION), offset=offset)
            return np.array(mmap_vector[0])
        except Exception as e:
            return f"An error occurred: {e}"

    def get_n_rows(self, row_num: int, n: int) -> np.ndarray:
        # This function loads a specified number of rows starting from row_num
        try:
            offset = row_num * DIMENSION * ELEMENT_SIZE
            mmap_vector = np.memmap(self.db_path, dtype=np.float32, mode='r', shape=(n, DIMENSION), offset=offset)
            return np.array(mmap_vector)
        except Exception as e:
            return f"An error occurred: {e}"


    def get_n_random_rows(self, indices) -> np.ndarray:
        try:
            min_idx = indices.min()
            max_idx = indices.max()
            offset = min_idx * DIMENSION * ELEMENT_SIZE
            n_rows = max_idx - min_idx + 1

            mmap_vector = np.memmap(
                self.db_path,
                dtype=np.float32,
                mode='r',
                shape=(n_rows, DIMENSION),
                offset=offset
            ) 
            relative_indices = indices - min_idx
            return np.array(mmap_vector[relative_indices])
        except Exception as e:
            return np.empty((len(indices), DIMENSION), dtype=np.float32)


    def get_all_rows(self) -> np.ndarray:
        # Take care this load all the data in memory
        num_records = self._get_num_records()
        vectors = np.memmap(self.db_path, dtype=np.float32, mode='r', shape=(num_records, DIMENSION))
        return np.array(vectors)

    # {
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #  }
    # def retrieve(self, query: Annotated[np.ndarray, (1, DIMENSION)], top_k=5):

    #     with open(self.index_path, 'rb') as index_file:
    #         cluster_mapping = pickle.load(index_file)

    #     nearest_neighbors_1 = []
    #     for centroid_1 in cluster_mapping.keys():
    #         distance = self._cal_score(query,centroid_1)
    #         nearest_neighbors_1.append((centroid_1,distance))

    #     nearest_neighbors_1 = sorted(nearest_neighbors_1, key=lambda x: -x[1])[:nprobe_1]

    #     nearest_neighbors_2 = []
    #     for centroid_1,_ in nearest_neighbors_1:
    #         dict = cluster_mapping[centroid_1]
    #         for centroid_2 in dict.keys():
    #             distance = self._cal_score(query,centroid_2)
    #             nearest_neighbors_2.append((centroid_2,distance))

    #     nearest_neighbors_2 = sorted(nearest_neighbors_2, key=lambda x: -x[1])[:nprobe_2]


    #     selected_ids = []
    #     id_distance_pairs = []
    #     # for centroid_2, _ in nearest_neighbors_2:
    #     #     for centroid_1, second_level_dict in cluster_mapping.items():

    #     #         if centroid_2 in second_level_dict:
    #     #             selected_ids.extend(second_level_dict[centroid_2])

    #     # indices = np.array(selected_ids)
    #     # rows = self.get_n_random_rows(indices)


    #     # id_distance_pairs = [
    #     #     (selected_ids[idx], self._cal_score(query, row))
    #     #     for idx, row in enumerate(rows)
    #     # ]

    #     # top_k_results = sorted(id_distance_pairs, key=lambda x: -x[1])[:top_k]

    #     # return [result[0] for result in top_k_results]

    #     for centroid_2, _ in nearest_neighbors_2:
    #         for centroid_1, second_level_dict in cluster_mapping.items():
    #             if centroid_2 in second_level_dict:
    #                 cluster_ids = second_level_dict[centroid_2]
    #                 cluster_ids_np = np.array(cluster_ids)
    #                 cluster_vectors = self.get_n_random_rows(cluster_ids_np)
    
    #                 # Calculate distances for vectors in the cluster
    #                 for idx, vector in enumerate(cluster_vectors):
    #                     distance = self._cal_score(query, vector)
    #                     id_distance_pairs.append((cluster_ids[idx], distance))

    #     # Sort and select top-k results
    #     top_k_results = sorted(id_distance_pairs, key=lambda x: -x[1])[:top_k]
    
    #     return [result[0] for result in top_k_results]


    # {
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #  }


    def retrieve(self, query: Annotated[np.ndarray, (1, DIMENSION)], top_k=5):
        with open(self.index_path, 'rb') as index_file:
            cluster_mapping = pickle.load(index_file)
            
        if self._get_num_records() == 10**6:
            available_ram = 20**6
        else: 
            available_ram = 50**6
            
        vector_size_bytes = DIMENSION * ELEMENT_SIZE  
        max_batch_size = available_ram // (vector_size_bytes * 500)
        
        # First-level clustering
        nearest_neighbors_1 = sorted(
            [(centroid_1, self._cal_score(query, centroid_1)) for centroid_1 in cluster_mapping.keys()],
            key=lambda x: -x[1]
        )[:nprobe_1]
    
        # Second-level clustering
        nearest_neighbors_2 = []
        for centroid_1, _ in nearest_neighbors_1:
            second_level_dict = cluster_mapping[centroid_1]
            for centroid_2 in second_level_dict.keys():
                nearest_neighbors_2.append((centroid_2, self._cal_score(query, centroid_2)))
    
        nearest_neighbors_2 = sorted(nearest_neighbors_2, key=lambda x: -x[1])[:nprobe_2] 
    
        top_k_heap = []
    
        for centroid_2, _ in nearest_neighbors_2: 
            for centroid_1, second_level_dict in cluster_mapping.items():
                if centroid_2 in second_level_dict:
                    cluster_ids = second_level_dict[centroid_2]
                    
                    cluster_ids_np = np.array(cluster_ids)
                    batch_size = min(len(cluster_ids), max_batch_size)
                    if batch_size == 0:
                        batch_size = 1
                        
                    # print(batch_size)
                    for i in range(0,len(cluster_ids),batch_size):
                        batch_ids = cluster_ids_np[i:i + batch_size]
                        cluster_vectors = self.get_n_random_rows(batch_ids)
                        
                        print(len(cluster_ids),len(batch_ids)) 
                        
                        for idx, vector in enumerate(cluster_vectors):
                            distance = self._cal_score(query, vector)
                            if len(top_k_heap) < top_k:
                                heapq.heappush(top_k_heap, (distance, batch_ids[idx]))
                            else:
                                heapq.heappushpop(top_k_heap, (distance, batch_ids[idx]))

        
        top_k_results = [heapq.heappop(top_k_heap)[1] for _ in range(len(top_k_heap))]
        top_k_results.reverse() 
    
        return top_k_results
        


    def _cal_score(self, vec1, vec2):
        dot_product = np.dot(vec1, vec2)
        norm_vec1 = np.linalg.norm(vec1)
        norm_vec2 = np.linalg.norm(vec2)
        cosine_similarity = dot_product / (norm_vec1 * norm_vec2)
        return cosine_similarity


    def _build_index(self):
        kmeans = MiniBatchKMeans(n_clusters=n_clusters_1, batch_size=batch_size_1, max_iter=200,n_init=10)

        vectors = self.get_all_rows()

        kmeans.fit(vectors)

        labels = kmeans.predict(vectors)
        centroids = kmeans.cluster_centers_

        cluster_mapping = {tuple(centroid): [] for centroid in centroids}

        for vector_id, label in enumerate(labels):
            centroid_key = tuple(centroids[label])
            cluster_mapping[centroid_key].append(vector_id)
        
        min_length=float('inf')
        for centroid,vector_ids in cluster_mapping.items():
            if len(vector_ids) < min_length and len(vector_ids) > 0:
                min_length = len(vector_ids)
        
        n_clusters_2 = min_length


        kmeans_2nd_level = MiniBatchKMeans(n_clusters=n_clusters_2, batch_size=batch_size_2, max_iter=200,n_init=10)

        cluster_mapping_1 = cluster_mapping
    
    
        for centroid_1 in cluster_mapping_1.keys():
        
            cluster_vector_ids = cluster_mapping_1[centroid_1]
            cluster_vector_ids_np = np.array(cluster_vector_ids)
    
            cluster_vectors = self.get_n_random_rows(cluster_vector_ids_np)
    
            labels = kmeans_2nd_level.fit_predict(cluster_vectors)
            centroids_2 = kmeans_2nd_level.cluster_centers_

    
            cluster_mapping_2 = {tuple(centroid_2): [] for centroid_2 in centroids_2}
    
            for vector_id, label in zip(cluster_vector_ids,labels):
                centroid_2_key = tuple(centroids_2[label])
                cluster_mapping_2[centroid_2_key].append(vector_id)
    
            cluster_mapping_1[centroid_1] = cluster_mapping_2
    
    
        # Save the cluster mapping to a file
        with open(self.index_path, 'wb') as index_file:
            pickle.dump(cluster_mapping_1, index_file)
            
    # {
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #  }


        
    # {
    #     "1st level cluster centroid":[centroids],
#         "1st level cluster centroid":[centroids],
#         "1st level cluster centroid":[centroids],
    # }


    # {  "1st level cluster centroid":  [ids] 
    # }  
    # 

In [136]:
import numpy as np
import os
import time
from dataclasses import dataclass
from typing import List
from memory_profiler import memory_usage
import gc

@dataclass
class Result:
    run_time: float
    top_k: int
    db_ids: List[int]
    actual_ids: List[int]

def run_queries(db, queries, top_k, actual_ids, num_runs):
    """
    Run queries on the database and record results for each query.

    Parameters:
    - db: Database instance to run queries on.
    - queries: List of query vectors.
    - top_k: Number of top results to retrieve.
    - actual_ids: List of actual results to evaluate accuracy.
    - num_runs: Number of query executions to perform for testing.

    Returns:
    - List of Result
    """
    global results
    results = []
    for i in range(num_runs):
        tic = time.time()
        db_ids = db.retrieve(queries[i], top_k)
        toc = time.time()
        run_time = toc - tic
        results.append(Result(run_time, top_k, db_ids, actual_ids[i]))
    return results

def memory_usage_run_queries(args):
    """
    Run queries and measure memory usage during the execution.

    Parameters:
    - args: Arguments to be passed to the run_queries function.

    Returns:
    - results: The results of the run_queries.
    - memory_diff: The difference in memory usage before and after running the queries.
    """
    global results
    mem_before = max(memory_usage())
    mem = memory_usage(proc=(run_queries, args, {}), interval = 1e-3)
    return results, max(mem) - mem_before

def evaluate_result(results: List[Result]):
    """
    Evaluate the results based on accuracy and runtime.
    Scores are negative. So getting 0 is the best score.

    Parameters:
    - results: A list of Result objects

    Returns:
    - avg_score: The average score across all queries.
    - avg_runtime: The average runtime for all queries.
    """
    scores = []
    run_time = []
    for res in results:
        run_time.append(res.run_time)
        # case for retireving number not equal to top_k, socre will be the lowest
        if len(set(res.db_ids)) != res.top_k or len(res.db_ids) != res.top_k:
            scores.append( -1 * len(res.actual_ids) * res.top_k)
            continue
        score = 0
        for id in res.db_ids:
            try:
                ind = res.actual_ids.index(id)
                if ind > res.top_k * 3:
                    score -= ind
            except:
                score -= len(res.actual_ids)
        scores.append(score)

    return sum(scores) / len(scores), sum(run_time) / len(run_time)

def get_actual_ids_first_k(actual_sorted_ids, k):
    """
    Retrieve the IDs from the sorted list of actual IDs.
    actual IDs has the top_k for the 20 M database but for other databases we have to remove the numbers higher than the max size of the DB.

    Parameters:
    - actual_sorted_ids: A list of lists containing the sorted actual IDs for each query.
    - k: The DB size.

    Returns:
    - List of lists containing the actual IDs for each query for this DB.
    """
    return [[id for id in actual_sorted_ids_one_q if id < k] for actual_sorted_ids_one_q in actual_sorted_ids]

This code to generate all the files for databases.

In [137]:
# def _write_vectors_to_file(vectors: np.ndarray, db_path) -> None:
#     mmap_vectors = np.memmap(db_path, dtype=np.float32, mode='w+', shape=vectors.shape)
#     mmap_vectors[:] = vectors[:]
#     mmap_vectors.flush()

# def generate_database(size: int) -> None:
#     rng = np.random.default_rng(DB_SEED_NUMBER)
#     vectors = rng.random((size, DIMENSION), dtype=np.float32)
#     return vectors

# vectors = generate_database(10**6)

# db_filename_size_20M = 'saved_db_20M.dat'
# if not os.path.exists(db_filename_size_20M): _write_vectors_to_file(vectors, db_filename_size_20M)
# db_filename_size_15M = 'saved_db_15M.dat'
# if not os.path.exists(db_filename_size_15M): _write_vectors_to_file(vectors[:15*10**6], db_filename_size_15M)
# db_filename_size_10M = 'saved_db_10M.dat'
# if not os.path.exists(db_filename_size_10M): _write_vectors_to_file(vectors[:10*10**6], db_filename_size_10M)
# db_filename_size_1M = 'saved_db_1M.dat'
# if not os.path.exists(db_filename_size_1M): _write_vectors_to_file(vectors[:1*10**6], db_filename_size_1M)

db_size=10**6
db = VecDB(db_size = db_size)

Code to generate the queries that will be used to evaluate the questions.

Note: QUERY_SEED_NUMBER will be changed at submission day

In [141]:
needed_top_k = 10000
rng = np.random.default_rng(QUERY_SEED_NUMBER)
query1 = rng.random((1, 70), dtype=np.float32)
query2 = rng.random((1, 70), dtype=np.float32)
query3 = rng.random((1, 70), dtype=np.float32)
query_dummy = rng.random((1, 70), dtype=np.float32)

actual_sorted_ids_20m_q1 = np.argsort(db.vectors.dot(query1.T).T / (np.linalg.norm(db.vectors, axis=1) * np.linalg.norm(query1)), axis= 1).squeeze().tolist()[::-1][:needed_top_k]
gc.collect()
actual_sorted_ids_20m_q2 = np.argsort(db.vectors.dot(query2.T).T / (np.linalg.norm(db.vectors, axis=1) * np.linalg.norm(query2)), axis= 1).squeeze().tolist()[::-1][:needed_top_k]
gc.collect()
actual_sorted_ids_20m_q3 = np.argsort(db.vectors.dot(query3.T).T / (np.linalg.norm(db.vectors, axis=1) * np.linalg.norm(query3)), axis= 1).squeeze().tolist()[::-1][:needed_top_k]
gc.collect()

queries = [query1, query2, query3]
actual_sorted_ids_20m = [actual_sorted_ids_20m_q1, actual_sorted_ids_20m_q2, actual_sorted_ids_20m_q3]

NameError: name 'db' is not defined

In [ ]:
# No more need to the actual vectors so delete it
del vectors
gc.collect()

This code to actually run the class you have been implemented. The `VecDB` class should take the database path, and index path that you provided.<br>
Note at the submission I'll not run the insert records. <br>
The query istelf will be changed at submissions day but not the DB

In [139]:
results = []
to_print_arr = []

In [140]:
# print("Team Number", TEAM_NUMBER)
# database_info = {
#     "1M": {
#         "database_file_path": db_filename_size_1M,
#         "index_file_path": PATH_DB_1M,
#         "size": 10**6
#     },
    # "10M": {
    #     "database_file_path": db_filename_size_10M,
    #     "index_file_path": PATH_DB_10M,
    #     "size": 10 * 10**6
    # },
    # "15M": {
    #     "database_file_path": db_filename_size_15M,
    #     "index_file_path": PATH_DB_15M,
    #     "size": 15 * 10**6
    # },
    # "20M": {
    #     "database_file_path": db_filename_size_20M,
    #     "index_file_path": PATH_DB_20M,
    #     "size": 20 * 10**6
    # }
# }

# for db_name, info in database_info.items():
# db = VecDB(database_file_path = info["database_file_path"], index_file_path = info["index_file_path"], new_db = False)

actual_ids = get_actual_ids_first_k(actual_sorted_ids_20m, db_size)
# Make a dummy run query to make everything fresh and loaded (wrap up)
res = run_queries(db, query_dummy, 5, actual_ids, 1)
# actual runs to evaluate
res, mem = memory_usage_run_queries((db, queries, 5, actual_ids, 3))
eval = evaluate_result(res)
to_print = f"score\t{eval[0]}\ttime\t{eval[1]:.2f}\tRAM\t{mem:.2f} MB"
print(to_print)
to_print_arr.append(to_print)
del db
del actual_ids
del res
del mem
del eval
gc.collect()

5566 457
5566 457
5566 457
5566 457
5566 457
5566 457
5566 457
5566 457
5566 457
5566 457
5566 457
5566 457
5566 82
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 457
7027 172
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 457
6878 23
6689 457
6689 457
6689 457
6689 457
6689 457
6689 457
6689 457
6689 457
6689 457
6689 457
6689 457
6689 457
6689 457
6689 457
6689 291
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 457
7661 349
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 457
6881 26
6683 457
6683 457
6683 457
6683 457
6683 457
6683 457
6683 457
6683 457
6683 457
6683 457
6683 457
6683 457
6683 457
6683 457
6683 285
7349 457
7349 457
7349 457
7349

0

In [ ]:
print("Team Number", TEAM_NUMBER)
print("\n".join(to_print_arr))